## 3.3 文本生成

对应 PDF **Problem (decoding)**：实现一个解码函数，支持提示词续写、最大生成长度、
温度缩放与 top-p（核）采样。

核心是一个循环：**每轮跑一次前向 → 取当前最后一个位置的分布 → 采样出一个 token → 拼回输入末尾**，直到碰到 `<|endoftext|>` 或到达长度上限。

In [30]:
import os
import sys

# 直接指定项目根目录，后面的路径一律相对它来写
os.chdir(r"C:\Users\intangible\Desktop\easy-llm")

# 注意切目录并不会让 import 找到 scripts/ 下的模块，还得把它加进模块搜索路径
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))

print("工作目录:", os.getcwd())
# 工作目录: C:\Users\intangible\Desktop\easy-llm

工作目录: C:\Users\intangible\Desktop\easy-llm


In [31]:
import torch

def decode(model, prompt, eot_token_id=None, max_new_tokens=256,
           temperature=0.5, top_p=0.9, context_length=256) -> list[int]:
    '''
    从语言模型采样生成，返回新生成的 token ID（包含 prompt）

    model: torch.nn.Module  输入 (batch, seq_len)，输出 (batch, seq_len, vocab_size)
    prompt: list[int]  已编码的 token ID 列表
    eot_token_id: int  结束符 <|endoftext|> 的 ID；None 表示不提前停止
    max_new_tokens: int  最多新生成的 token 数
    temperature: float  温度参数 τ，越小分布越尖锐（趋近贪心），越大越平缓
    top_p: float  Top-p 核采样阈值，只保留累计概率达到 p 的那一小撮候选
    context_length: int  模型最大上下文长度，每一步只保留最后这么多个 token 作为输入
    '''
    ids = list(prompt)                               # 提示词长度为 prompt_length
    x = torch.tensor(ids).unsqueeze(0)               # 补上 batch 维：(seq_len,) -> (1, seq_len) 
    model.eval()
    torch.set_grad_enabled(False)                    # 全局开关，关闭梯度计算，不再构建计算图

    # 每轮产出一个 token：前向 → 取最后一个位置上的分布 → 采样 → 拼回输入
    for i in range(1, max_new_tokens + 1):          # 预测第 prompt_length + i 个位置上的 token（i 从 1 开始）
        x_input = x[:, -context_length:]            # 从倒数第 context_length 个元素开始，一直取到结尾；若 len(x) < context_length，就从第一个元素开始；保证输入长度不超过模型的上下文窗口
        pred_y = model(x_input)[:, -1, :]           # (1, vocab_size) 只要当前最后一个位置上的预测

        # 温度缩放：τ→0 时最大的 logit 一枝独秀（趋近贪心），τ 越大分布越平、越敢选低概率词
        probs = torch.softmax(pred_y / temperature, dim=-1).squeeze(0)   # (1, vocab_size) -> (vocab_size,)

        # top-p：先按概率从大到小排序并求前缀和，累计到 p 的那一段就是要保留的"核"
        sorted_probs, sorted_indices = probs.sort(descending=True) # 同时返回 (排序后的序列， 排序后的序列元素的原始下标/ token id)
        cumsum = sorted_probs.cumsum(dim=-1) # (vocab_size,)
        cutoff_idx = torch.searchsorted(cumsum, top_p)   # 找到第一个使前缀和 >= p 的下标，假设是 j；
        sorted_mask = torch.arange(cumsum.shape[-1], device=probs.device) <= cutoff_idx # 构建掩码数组，0~j-1 号元素为 true，剩余元素为 false
        # 0 1 2 ... vocab_size

        # mask 是按"排序后的顺序"创建的，需要通过 sorted_indices 映射回原始下标，再作用到原始顺序的 probs 上
        # ~sorted_mask 将非 top-p 位置设置为 true，再使用 sorted_indices[~sorted_mask] 仅保留 top-p 屏蔽的那些元素对应的下标，方便后续将这些下标对应的 probs 值置 0
        probs[sorted_indices[~sorted_mask]] = 0

        probs = probs / probs.sum()                  # 丢掉尾部之后要重新归一化
        next_token_id = torch.multinomial(probs, num_samples=1).item() # 按给定概率 probs 采样 num_samples 个元素（默认不放回采样，即采样出的多个元素互不相同）

        ids.append(next_token_id) # 将新生成的 token 加入到输入序列中（列表形式）
        x = torch.cat([x, torch.tensor([[next_token_id]], device=x.device)], dim=1) # 将新生成的 token 加入到输入序列中（张量形式）
        if eot_token_id is not None and next_token_id == eot_token_id: # 生成了结束符，解码提前终止
            break

    return ids

In [32]:
import torch

# ---------- 例子 1：温度 τ 怎么改变采样分布 ----------
logits = torch.tensor([2.0, 1.0, 0.5, 0.2, -1.0])   # 5 个候选 token 的 logits

for tau in [0.1, 0.5, 1.0, 2.0]:
    p = torch.softmax(logits / tau, dim=-1)
    print(f"τ={tau:<4} → 概率 {[round(v, 3) for v in p.tolist()]}")
# τ=0.1  → [1.0,   0.0,   0.0,   0.0,   0.0  ]   ← 几乎 one-hot，等价于贪心
# τ=1.0  → [0.554, 0.204, 0.124, 0.092, 0.028]   ← 原始 softmax
# τ=2.0  → [0.369, 0.224, 0.174, 0.150, 0.082]   ← 明显被拉平，尾部候选机会变大



τ=0.1  → 概率 [1.0, 0.0, 0.0, 0.0, 0.0]
τ=0.5  → 概率 [0.823, 0.111, 0.041, 0.022, 0.002]
τ=1.0  → 概率 [0.554, 0.204, 0.124, 0.092, 0.028]
τ=2.0  → 概率 [0.369, 0.224, 0.174, 0.15, 0.082]


In [33]:
# ---------- 例子 2：top-p 保留多少个候选 ----------
probs = torch.softmax(logits, dim=-1)               # 以 τ=1 的分布为例
for top_p in [0.5, 0.9, 1.0]:
    sorted_probs, _ = probs.sort(descending=True)
    cumsum = sorted_probs.cumsum(dim=-1)
    cutoff = torch.searchsorted(cumsum, top_p).item()
    print(f"top_p={top_p} → 保留前 {cutoff + 1} 个候选，其累计概率 {cumsum[cutoff]:.3f}")

top_p=0.5 → 保留前 1 个候选，其累计概率 0.554
top_p=0.9 → 保留前 4 个候选，其累计概率 0.972
top_p=1.0 → 保留前 5 个候选，其累计概率 1.000


### 例子 3：载入真实的检查点与分词器

例子 1、2 只是示例，而非实际采样过程。要看到真正「会说话」的模型，还需要：

| 组件 | 本机位置 |
|---|---|
| 训练好的分词器（文本 ↔ token ID） | `results/tokenizer/TinyStoriesV2-GPT4-train/{vocab.json, merge.txt}` |
| 训练好的权重 | `results/checkpoints/TinyStoriesV2-GPT4/TinyStoriesV2-GPT4-bs64-lr1.0e-02_1.0e-03-s2026-iter004999.pt` |

这个检查点是 3.2 节在 TinyStories 上跑 5000 步的产物：10,000 词表、`d_model=512`、4 层、16 头、
`d_ff=1344`（门控 FFN）、RoPE，共 22.70 M 参数，在第 4999 步保存。

整条链路是 **文本 → 分词器 `encode` → `decode()` 采样 → 分词器 `decode` → 文本**，

In [34]:
from bpe_tokenizer import Tokenizer      # 补充 1 里即将实现的字节级 BPE 分词器

# 项目根目录的相对路径
TOKENIZER_DIR = "results/tokenizer/TinyStoriesV2-GPT4-train"

# 用法就两个方向：encode 把文本变成 token ID，decode 把 token ID 还原成文本
tokenizer = Tokenizer()
tokenizer.from_files(f"{TOKENIZER_DIR}/vocab.json",
                     f"{TOKENIZER_DIR}/merge.txt",
                     ["<|end_of_text|>"])     # 训练时注册的特殊 token，当时写的时候搞错了，原始数据集/语料里的分隔符实际是 "<|endoftext|>"

print("词表大小:", len(tokenizer.vocab))
ids = tokenizer.encode("Once upon a time") # encode() 方法将字符串转换成 token id
print("encode:", ids)
print("decode:", tokenizer.decode(ids)) # decode() 方法将模型输出的 token id 转回字符串

# 结束符 ID：分词器注册的特殊 token 是 "<|end_of_text|>"，而语料里的分隔符写的是 "<|endoftext|>"，两者并非同一个 token，所以模型实际生成的是后者，这里按注册的名字取 ID。
EOT_ID = tokenizer.encode("<|end_of_text|>")[0]
print("EOT_ID:", EOT_ID)

# encode: [437, 446, 259, 403]     ← 分别对应 b'Once' / b' upon' / b' a' / b' time'

词表大小: 10000
encode: [437, 446, 259, 403]
decode: Once upon a time
EOT_ID: 256


In [39]:
import torch
from transformer import TransformerLM
from training_utils import load_checkpoint     # 2.3.2 实现的加载函数：torch.load + load_state_dict
import os


#ckpt = "bs64-lr1.0e-02_1.0e-03-s2026-iter000500.pt" # 早期训练版 500 步
ckpt = "bs64-lr1e-2_1e-3-s2026-ablation_no_gated-iter004999.pt" # FFN 无门控消融版（val 损失为 1.47，比完整版低 0.02） 

#ckpt = "bs64-lr1.0e-02_1.0e-03-s2026-iter004999.pt" # 完整版的最优模型（Val 损失为 1.49

CKPT_PATH = (os.path.join("results/checkpoints/TinyStoriesV2-GPT4/", ckpt))

# # 模型结构必须与训练时严格一致，否则 load_state_dict 会报形状不匹配
# model = TransformerLM(vocab_size=10000, context_length=256, d_model=512,
#                       num_layers=4, num_heads=16, d_ff=1344, rope_theta=10000.0)

# FFN 无门控消融版的模型结构
d_ff = int(512 * 4 + 63) // 64 * 64
model = TransformerLM(vocab_size=10000, context_length=256, d_model=512,
                      num_layers=4, num_heads=16, d_ff=1344, ffn_gated=False, rope_theta=10000.0)

iter_num = load_checkpoint(CKPT_PATH, model)
#iter_num = -1
model.eval()

print(f"已载入第 {iter_num} 步的检查点，参数量 {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")

prompt = "Once upon a time, there was a little girl named Lily."
prompt_ids = tokenizer.encode(prompt)

torch.manual_seed(0)                           # 固定采样随机数，让下面这段输出可以复现
out_ids = decode(model, prompt_ids, eot_token_id=EOT_ID, max_new_tokens=256,
                 temperature=0.8, top_p=0.9, context_length=256)
print(tokenizer.decode(out_ids))

已载入第 4999 步的检查点，参数量 19.94 M
Once upon a time, there was a little girl named Lily. She had a big, soft blanket that she loved very much. One day, she went to the park to play with her friends.At the park, Lily saw a little boy who was sad. He lost his toy. Lily wanted to help him. She said, "Don't worry, I will help you find your toy."Lily and the boy looked for the toy together. They looked under the trees and behind the bushes. At last, they found the toy near a big tree. Lily was so happy! She said, "Thank you, Lily!"The boy smiled and said, "You're welcome, Lily! Let's play together again!" They played all day, and Lily was not sad anymore. She knew that her toy and her friends would always be there to help her.<|endoftext|>Once upon a time, there was a big, dark room. In the room, there was a little boy named Tim. Tim was a very dependable boy. He always listened to his mom and dad.One day, Tim's mom asked him to help her clean the room. Tim did not want to do it. He wanted to play

In [ ]:
# Once upon a time, there was a little girl named Lily. She had a big, soft blanket that she loved
# very much. One day, she went to the park to play with her friends.At the park, Lily met a boy
# named Tom. Tom liked to tease Lily and make her jump over a stick. Lily said, "Tom, let's play a
# game. I will give you a ball." Tom liked the idea and they played together.Lily said, "Tom, I will
# throw the ball to you, and you will catch it." Tom threw the ball, and Lily caught it. They
# played for a long time, and Tom got very
#   ——5,000 步、22.7 M 参数的小模型已经能写出语法通顺、情节连贯的小故事。
#     这段没写到结尾，达到 max_new_tokens=120 的输出上限停的。

In [40]:
# 同一个提示词，只动采样参数：τ 决定分布有多平，top_p 决定砍掉多长的尾部
# 这里只打印新生成的部分，所以用 out_ids[len(prompt_ids):] 把提示词本身去掉
for tau, p in [(0.2, 0.9), (0.8, 0.9), (1.5, 0.95)]:
    torch.manual_seed(0)                       # 每个组合用同一个种子，方便横向对比
    out_ids = decode(model, prompt_ids, eot_token_id=EOT_ID, max_new_tokens=45,
                     temperature=tau, top_p=p, context_length=256)
    print(f"τ={tau}, top_p={p}")
    print("   ", tokenizer.decode(out_ids[len(prompt_ids):]), "\n")


τ=0.2, top_p=0.9
     She loved to play with her toys and her dog, Max. One day, Lily found a big box in her room. She was very excited and wanted to see what was inside.Lily opened the box and found a pretty 

τ=0.8, top_p=0.9
     She had a big, soft blanket that she loved very much. One day, she went to the park to play with her friends.At the park, Lily saw a little boy who was sad. He lost his toy. 

τ=1.5, top_p=0.95
     She had a big birdcage in her room. One day, Lily wanted to take a birdcage of her smooth Abi.On few vests, Lily set notice that herZoe Reakedfange and five di Oliver left her bird friends 



In [ ]:
# τ=0.2, top_p=0.9
#      She loved to play with her toys and have fun with her friends. One day, she found a big,
#      red ball in her yard. She was very happy and wanted to play with it.Lily took the ball to her
#                                                              ← 分布最尖锐，续写最保守、最贴着提示词
# τ=0.8, top_p=0.9
#      She had a big, soft blanket that she loved very much. One day, she went to the park to play
#      with her friends.At the park, Lily met a boy named Tom. Tom liked to tease Lily and make
#                                                              ← 有变化但仍然通顺
# τ=1.5, top_p=0.95
#      She had a big birdcage. Every night, it gave dark feathers...hel polarss control of her
#      babiesbbist - Sophie was even hopeful.One morning uncomfortableZoe Reannafange and carry away,
#      left her birdSm     ← τ 太大把长尾候选也放进来了，开始出现生造词